# ARE-006: interactive recommendation exploration

Change the parameters below, then rerun the relevant sections. Changing ALS parameters requires explicit retraining. Changing user_id or top_k only recalculates recommendations and diagnostics. All preparation, inference and metrics live in src/. No test data is read.

Privacy: outputs can contain personal histories. Do not save executed outputs; clear all outputs and execution counts before saving. Optional YummyAnime responses remain in memory only.

In [ ]:
# Model parameters: require retrain=True to create a matching saved model.
als_factors = 64
regularization = 0.1
iterations = 20
seed = 42
retrain = False  # Explicit one-time action. Return to False after training.

# Inference parameters: no retraining needed.
top_k = 10
validation_user_id = None  # None selects the first validation user; see available IDs below.
yummyanime_nickname = None  # Optional: your own public profile nickname. None makes zero requests.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
from anime_recommender.exploration import project_root
from anime_recommender.recommendation_exploration import (
    load_workspace, validation_view, validation_diagnostics, optional_profile_view,
)

root = project_root()

## Load a model, or explicitly train

Run this cell only when loading/changing the model. A mismatch raises an informative error; it never silently trains. Nondefault models are saved under an ignored separate notebook directory, preserving the baseline. Training uses only the prepared train CSR. When changing only user_id/top_k, rerun the sections below this one.

In [ ]:
workspace = load_workspace(root, factors=als_factors, regularization=regularization,
                           iterations=iterations, seed=seed, retrain=retrain)
display(workspace.bundle.metadata)
display(workspace.inputs.validation_users[["dataset_user_id"]])

## Selected validation user: observed context

Context shows original ratings, titles, is_positive and is_warm. Every observed context item is excluded from both models, including low/zero ratings; only warm positive context goes into ALS fold-in. To select another user, edit validation_user_id above, rerun the parameter cell and continue here. Keep retrain=False.

In [ ]:
selected = validation_view(workspace, user_id=validation_user_id, top_k=top_k)
display(selected["user_id"])
display(selected["context"])
display(selected["statuses"])

## Hidden positive targets: diagnostics only

Recommendations have already been generated using context alone. Targets are subsequently used for is_hit annotations and metric computation. Cold targets are retained.

In [ ]:
display(selected["positive_targets"])

## Top-N from both models

Titles, rank, score, train popularity (unique_users) and is_hit. ALS scores are dot products, not probabilities. ALS-only is unavailable without warm positives; this validation comparison never silently adds fallback.

In [ ]:
display(selected["recommendations"])

## Selected-user metrics and target hits

Users without positive targets have undefined Recall/NDCG rather than zero. These are diagnostics, not parameter tuning.

In [ ]:
display(selected["metrics"].T)
display(selected["hits"])
selected["hits"].plot.bar(y="target_hits", legend=False, title="Target hits for selected user")
plt.ylabel("Hits")
plt.tight_layout()
plt.show()

## ALS score distribution

Scores over the full train vocabulary for the selected user's fold-in vector, before seen-item filtering. Empty distribution means ALS has no warm positive context.

In [ ]:
display(selected["scores"].describe())
selected["scores"].plot.hist(bins=30, title="ALS scores (not probabilities)")
plt.xlabel("Dot-product score")
plt.tight_layout()
plt.show()

## Popularity of recommended anime

Both models use the same train-only popularity counts, even though ALS ranks by personalized scores. Each point/bar represents the item at that recommendation rank.

In [ ]:
selected["recommendations"].pivot(index="rank", columns="model_name", values="unique_users").plot.bar(
    title="Train popularity of recommended items")
plt.ylabel("Unique positive train users")
plt.tight_layout()
plt.show()

## Cohort-level validation comparison at the chosen K

Rerun after changing top_k; changing only the selected user does not require rerunning this section. Full Recall includes cold targets; Warm Recall conditions on warm context and nonempty warm target. NDCG's ideal denominator also includes cold targets. Every metric denominator is displayed. No test evaluation or hyperparameter search.

In [ ]:
validation = validation_diagnostics(workspace, top_k=top_k)
display(validation["summary"].T)
validation["summary"].set_index("model_name")[["full_recall_at_k", "ndcg_at_k"]].plot.bar(
    title=f"Validation Recall@{top_k} and NDCG@{top_k}")
plt.ylabel("Macro average")
plt.tight_layout()
plt.show()

## Optional YummyAnime experiment

Set yummyanime_nickname, rerun the parameter cell, then this section. At most three requests: profile, lists, favorites. Only public X-Application token from local .env/environment; no owner Bearer token. Errors stop the experiment and are shown without response bodies or secrets.

Binary positive policy: rating>=5 (valid scale 1..10), completed without a rating (0/missing), or favorite. Planned and dropped veto positivity even if rated/favorite. Warm positives only enter fold-in. Every mapped known item is excluded from both ALS and fallback, regardless of rating/status. Popularity fallback is used only when the successfully fetched history has no warm positives. Unknown MAL IDs cannot be identified in the model; mapping gaps and available fold-in interactions are reported.

No profile history is written to disk. Clear outputs before saving the notebook.

In [ ]:
profile = optional_profile_view(workspace, nickname=yummyanime_nickname,
                                top_k=top_k, env_path=root / ".env")
display(profile["state"], profile["error"])
display(profile["requests"])
display(profile["coverage"])
display({"fallback": profile.get("fallback"), "reason": profile.get("fallback_reason")})
display(profile["recommendations"])

## Limitations

A small sampled dataset and random context/target split without timestamps do not measure future chronological performance. No automatic tuning is performed here. YummyAnime status/favorite signals differ from the rating-only training signal, so this remains an experimental adapter. Profile API failures do not trigger inference from partial data. Run the tests and clear all outputs before review.